In [ ]:
# Setup: credentials from .env (never hardcode)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from notebooks.load_env import load_env, get_project_root
load_env()
PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "archive"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# Load chunk data (Colab: drive path; local: project path)
import pandas as pd
df = pd.read_parquet("drive/MyDrive/pilot_chunks_with_embeddings.parquet" if IN_COLAB else PROJECT_ROOT / "notebooks" / "pilot_chunks_with_embeddings.parquet", engine="fastparquet")

# 02 — Prepare Neo4j Graph Data

Entity extraction and relationship ingestion. **Prerequisites:** `.env` configured, `docker compose up -d`.

---

## 1. Entity Extraction

In [ ]:
# Show what's in the drive directory
import pandas as pd
df = pd.read_parquet("drive/MyDrive/pilot_chunks_with_embeddings.parquet", engine="fastparquet")

# Create a new column 'article_chunk_id' as a unique identifier per (article_id, chunk_id) combination
# First, assign a new chunk_id within each article group, then create the combined id
df = df.sort_values(['article_id'])  # optional: sort for reproducibility
df['chunk_id'] = df.groupby('article_id').cumcount()
df['article_chunk_id'] = df['article_id'].astype(str) + "_" + df['chunk_id'].astype(str)
df

In [ ]:
ENTITY_LABELS_WITH_DESC = {
    "ORGANIZATION": "Corporate entities, brands, or manufacturers like Apple, Tesla, or TSMC.",
    "SERVICE_PROVIDER": "Firms providing market analysis, logistics, or consulting like RationalStat.",
    "TECHNOLOGY": "Broad technical standards or innovations like 5G, 2nm silicon, or Lidar.",
    "COMPONENT": "Physical sub-parts used in manufacturing like power semiconductors or battery cells.",
    "MARKET_METRIC": "Quantitative data including market size (USD), CAGR, or market share percentages.",
    "INDUSTRY_VERTICAL": "Specific market sectors such as the EV industry or semiconductor foundries.",
    "LOCATION": "Geographic regions, cities, or countries where market activity occurs.",
    "PERSON": "Named individuals, typically executives, analysts, or key stakeholders (e.g., Tim Cook, Jensen Huang).",
    "PRODUCT": "Specific consumer or commercial goods ready for market, such as the iPhone 15, iOttie Wireless Duo, or Model 3."
}


from gliner import GLiNER

# Load GLiNER model for NER
model = GLiNER.from_pretrained("knowledgator/gliner-x-small-v0.5")

from tqdm import tqdm

# Map your entity labels to the GLiNER label format (lowercase, no spaces)
gliner_labels = [label.lower() for label in ENTITY_LABELS_WITH_DESC.keys()]

# Optionally: Prepare type descriptions for GLiNER, if supported (check doc: https://github.com/urchade/gliner)
# Prepare gliner_labels_with_desc if GLiNER supports descriptions, else fallback to just gliner_labels
# As of v0.5, GLiNER does support label descriptions ("descriptions" arg)

# Prepare the entity labels and their descriptions as required by GLiNER
gliner_label_desc = [
    {"label": label.lower(), "description": desc}
    for label, desc in ENTITY_LABELS_WITH_DESC.items()
]

# Determine device
import torch
model_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device for GLiNER inference: {model_device}")

gliner_labels = [
    "organization",
    "service_provider",
    "technology",
    "component",
    "market_metric",
    "industry_vertical",
    "location",
    "person",
    "product"
]

# Prepare a results list to store NER results for each chunk
ner_results = []

# Apply NER model to each row's chunk_text
for idx, row in df.iterrows():
    text = row["chunk_text"]
    article_chunk_id = row["article_chunk_id"]
    try:
        # Ensure model runs on GPU if available
        import torch
        model_device = "cuda" if torch.cuda.is_available() else "cpu"
        entities = model.predict_entities(text, gliner_labels, threshold=0.5, device=model_device)
        # for each entity, add extra info
        for entity in entities:
            result = {
                "article_chunk_id": article_chunk_id,
                "entity": entity["text"],
                "type": entity["label"].upper(),  # map to the format of your types
                "start": entity.get("start", None),
                "end": entity.get("end", None),
                "confidence": entity.get("score", None),
                "chunk_index": row["chunk_id"]
            }
            ner_results.append(result)
    except Exception as e:
        print(f"NER failed for chunk {article_chunk_id}: {e}")

ner_df = pd.DataFrame(ner_results)
ner_df.head()

In [ ]:
# Check CUDA (GPU) availability before proceeding
import torch

if torch.cuda.is_available():
    print("CUDA (GPU) is available. Running on:", torch.cuda.get_device_name(0))
else:
    print("CUDA (GPU) is NOT available. Model will run on CPU.")
    print("If this is Colab, ensure you have selected GPU as the runtime (Runtime > Change runtime type > Hardware accelerator: GPU).")


In [ ]:
# If you are using Cursor/VSCode with a Colab kernel only (not using the Colab web UI directly):
# - You cannot select the GPU from within the notebook; you must set the hardware accelerator in Colab before connecting the kernel.
print("To use a GPU with a Colab kernel in Cursor/VSCode, follow these steps:")
print("1. Open https://colab.research.google.com/ in your browser.")
print("2. Click 'Runtime' > 'Change runtime type'.")
print("3. Set 'Hardware accelerator' to 'GPU' and click 'Save'.")
print("4. Then, connect your Cursor/VSCode to this Colab kernel session.")
print()
print("Uploading files: You must upload necessary files (like data files) to the Colab environment.")
print("You can do this via Google Drive or by running upload code in the notebook.")
print("To upload interactively in a code cell (works if Colab kernel allows UI popups):")
print("  from google.colab import files")
print("  uploaded = files.upload()")
print()
print("Alternatively, mounting Google Drive is recommended for large files:")
print("  from google.colab import drive")
print("  drive.mount('/content/drive')")

In [ ]:
import pandas as pd

mock_extraction_data = [
    # Supply Chain & Typos (Tests Normalization)
    {"head": "Nvidia", "head_type": "organization", "relation": "supplies", "tail": "TSMC", "tail_type": "organization", "chunk_id": "ch_001"},
    {"head": "NVIDIA Corp", "head_type": "organization", "relation": "produces", "tail": "H100 GPU", "tail_type": "product", "chunk_id": "ch_002"},
    
    # Tech & Ownership
    {"head": "TSMC", "head_type": "organization", "relation": "produces", "tail": "3nm Process", "tail_type": "technology", "chunk_id": "ch_003"},
    {"head": "Apple", "head_type": "organization", "relation": "supplies", "tail": "TSMC", "tail_type": "organization", "chunk_id": "ch_004"},
    
    # Competition & Geography
    {"head": "Intel", "head_type": "organization", "relation": "competes with", "tail": "Nvidia", "tail_type": "organization", "chunk_id": "ch_005"},
    {"head": "Intel", "head_type": "organization", "relation": "located in", "tail": "Santa Clara", "tail_type": "location", "chunk_id": "ch_006"},
    
    # Market Metrics (Linking to entities)
    {"head": "Nvidia", "head_type": "organization", "relation": "produces", "tail": "$2.2T Valuation", "tail_type": "market metric", "chunk_id": "ch_007"},
    
    # Person & Events (Hyper-linking)
    {"head": "Jensen Huang", "head_type": "person", "relation": "produces", "tail": "GTC Keynote", "tail_type": "product", "chunk_id": "ch_008"}
]

df_mock = pd.DataFrame(mock_extraction_data)

In [ ]:
import os
from neo4j import GraphDatabase
neo4j_driver = GraphDatabase.driver(os.getenv("NEO4J_URI", "bolt://localhost:7687"), auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD")))

In [ ]:
import pandas as pd
import re
from neo4j import GraphDatabase

# --- 1. DATA PREP & NORMALIZATION ---

def normalize_for_id(text):
    if not text or pd.isna(text): return "unknown"
    text = str(text).lower().strip()
    # Remove corporate suffixes
    text = re.sub(r'\b(inc|corp|llc|ltd|corporation|incorporated)\b', '', text)
    # Remove special characters
    text = re.sub(r'[^\w\s]', '', text)
    return "_".join(text.split())

def prepare_dataframe(df):
    """Generates IDs and cleans the dataframe for Neo4j consumption."""
    df_new = df.copy()
    df_new['head_id'] = df_new['head'].apply(normalize_for_id)
    df_new['tail_id'] = df_new['tail'].apply(normalize_for_id)
    
    # Filter out invalid rows (NaNs or "unknown" IDs)
    mask = (
        df_new["head_id"].notnull() & 
        (df_new["head_id"] != "unknown") &
        (df_new["tail_id"] != "unknown") &
        (df_new["head_id"] != "")
    )
    return df_new[mask]

# --- 2. THE UPSERT FUNCTION ---

def upsert_graph_data(driver, dataframe):
    """
    Principal-level upsert: Handles string-casting, 
    dynamic labels, and relationship batching.
    """
    # Define Queries inside the function scope
    upsert_nodes_query = """
    UNWIND $batch as row
    CALL apoc.merge.node([row.head_type], {id: row.head_id}, {name: row.head}, {}) YIELD node as h
    CALL apoc.merge.node([row.tail_type], {id: row.tail_id}, {name: row.tail}, {}) YIELD node as t
    RETURN count(*)
    """

    upsert_rels_query = """
    UNWIND $batch as row
    MATCH (h {id: row.head_id})
    MATCH (t {id: row.tail_id})
    CALL apoc.merge.relationship(h, row.relation_type, {}, {chunk_id: row.chunk_id}, t) YIELD rel
    RETURN count(*)
    """

    # Sanitize data for Neo4j (Convert all to string, handle NaNs)
    data_list = dataframe.to_dict('records')
    clean_batch = []
    for row in data_list:
        clean_row = {
            k: (str(v) if pd.notnull(v) else "") 
            for k, v in row.items()
        }
        # Pre-process relation types (e.g., "partners with" -> "PARTNERS_WITH")
        clean_row['relation_type'] = clean_row['relation'].replace(" ", "_").upper()
        clean_batch.append(clean_row)

    # Execute
    with driver.session() as session:
        # Pass 1: Nodes
        session.run(upsert_nodes_query, batch=clean_batch)
        print(f"Successfully upserted {len(clean_batch)} node pairs.")

        # Pass 2: Relationships
        session.run(upsert_rels_query, batch=clean_batch)
        print(f"Successfully upserted {len(clean_batch)} relationships.")

# --- 3. EXECUTION FLOW ---

# 1. Prep the data
df_ready = prepare_dataframe(df_mock)

# 2. Push to Neo4j
neo4j_driver = GraphDatabase.driver(os.getenv("NEO4J_URI", "bolt://localhost:7687"), auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD")))
# neo4j_driver = GraphDatabase.driver(os.getenv("NEO4J_URI", "bolt://localhost:7687"), auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD")))
upsert_graph_data(neo4j_driver, df_ready)

### Lets upsert the real data contain entity extract and relationships between entities

In [ ]:
import pandas as pd



df_ner = pd.read_csv(DATA_DIR / "ner_final_results.csv")
df_rel = pd.read_csv(DATA_DIR / "rel_final_results.csv")
print("NER table loaded:", df_ner.shape)
print("REL table loaded:", df_rel.shape)

In [ ]:
import pandas as pd
import re

# 1. LOAD DATA
# Adjust filenames as needed


# 2. BLACKLIST (The 'Stop-Entities')
# Purging generic nouns that create "Supernodes" and pollute graph reasoning
BLACKLIST = {
    'The Company', 'Management', 'Our Team', 'Fiscal Year', 'Quarter', 
    'Products', 'Services', 'Industry', 'Today', 'Yesterday', 'Someone',
    'Private Sector', 'Public Sector', 'Executive', 'Analyst', 'Market'
}

# 3. METRICS: START
ner_count_start = len(df_ner)
ner_unique_start = df_ner['entity'].nunique()

# 4. CLEANING & BLACKLIST FUNCTION
def clean_and_check(name):
    if not isinstance(name, str) or len(str(name)) < 2: 
        return None
    
    # Surgical Regex: Remove leading/trailing punctuation (commas, dots, etc.)
    name = re.sub(r'^[\s,.\-()]+|[\s,.\-()]+$', '', name)
    
    # Suffix Stripping (Inc, Corp, etc.)
    suffixes = r'\s+(Inc|Corp|Ltd|LLC|GmbH|Co|Corporation|Limited|Incorporated)\.?$'
    name = re.sub(suffixes, '', name, flags=re.IGNORECASE).strip()
    
    # Case Normalization (Standardize to Title Case, preserve short acronyms)
    if name.isupper() and len(name) <= 5:
        normalized = name.strip()
    else:
        normalized = name.strip().title()
        
    # Final check against Blacklist
    if normalized in BLACKLIST:
        return None
    return normalized

# 5. EXECUTE CLEANING
df_ner['canonical_name'] = df_ner['entity'].apply(clean_and_check)

# 6. CONFIDENCE & RELATIONSHIP SYNC PIPELINE
# Step A: Filter Relations by High Confidence (0.90)
df_rel_filtered = df_rel[df_rel['confidence'] >= 0.90].copy()

# Step B: Create Mapping {Original -> Cleaned}
# We drop None values (blacklisted) here to ensure they don't sync to REL
name_map = df_ner.dropna(subset=['canonical_name']).set_index('entity')['canonical_name'].to_dict()

# Step C: Sync REL table with Cleaned Names
df_rel_filtered['head_clean'] = df_rel_filtered['head'].map(name_map)
df_rel_filtered['tail_clean'] = df_rel_filtered['tail'].map(name_map)

# Step D: Drop relations where either head or tail was blacklisted or missing
df_rel_final = df_rel_filtered.dropna(subset=['head_clean', 'tail_clean']).copy()

# Step E: Protect entities that are part of these high-signal relationships
rel_participants = set(df_rel_final['head_clean']).union(set(df_rel_final['tail_clean']))

# Step F: Final NER Filter (High Confidence OR Relationship Participant)
# Note: We only keep rows where 'canonical_name' is not None (passed blacklist)
df_ner_final = df_ner[
    (df_ner['canonical_name'].notna()) & 
    ((df_ner['confidence'] >= 0.90) | (df_ner['canonical_name'].isin(rel_participants)))
].copy()

# 7. METRICS: END
ner_count_end = len(df_ner_final)
ner_unique_end = df_ner_final['canonical_name'].nunique()
rel_count_end = len(df_rel_final)

# 8. DATA INTEGRITY REPORT
print("="*60)
print("PRINCIPAL ENGINEER DATA PIPELINE REPORT")
print("="*60)
print(f"NER ROWS:          {ner_count_start:,}  ->  {ner_count_end:,}")
print(f"UNIQUE ENTITIES:   {ner_unique_start:,}  ->  {ner_unique_end:,}")
print(f"RELATIONSHIPS:     {len(df_rel):,}  ->  {rel_count_end:,}")
print("-" * 30)
complexity_reduction = (1 - (ner_unique_end / ner_unique_start)) * 100
print(f"Complexity Reduction: {complexity_reduction:.2f}%")
print(f"High-Conf Rel Participation: {len(rel_participants):,} entities")
print("="*60)

# 9. SAVE FINAL ASSETS
df_ner_final['entity'] = df_ner_final['canonical_name']
df_rel_final['head'] = df_rel_final['head_clean']
df_rel_final['tail'] = df_rel_final['tail_clean']

# Identify which columns actually exist to avoid KeyError
# We check for common variations like 'entity_type', 'type', or 'label'
potential_ner_cols = ['article_chunk_id', 'entity', 'entity_type', 'type', 'label', 'confidence']
ner_cols_to_save = [c for c in potential_ner_cols if c in df_ner_final.columns]

potential_rel_cols = ['article_chunk_id', 'head', 'relation', 'tail', 'confidence']
rel_cols_to_save = [c for c in potential_rel_cols if c in df_rel_final.columns]

# Use the dynamic lists for exporting
df_ner_final[ner_cols_to_save].to_csv(DATA_DIR / "ner_cleaned.csv", index=False)
df_rel_final[rel_cols_to_save].to_csv(DATA_DIR / "rel_cleaned.csv", index=False)

print("="*60)
print("SUCCESS: CLEANED DATA EXPORTED")
print(f"NER path: .../data/archive/ner_cleaned.csv")
print(f"REL path: .../data/archive/rel_cleaned.csv")
print(f"Exported NER columns: {ner_cols_to_save}")
print("="*60)

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PWD = os.getenv("NEO4J_PASSWORD")

def normalize_id(text):
    if not text or pd.isna(text): return "unknown"
    return "_".join(str(text).lower().strip().split())

def run_neo4j_rel_ingestion():
    ner_path = str(DATA_DIR / "ner_cleaned.csv")
    rel_path = str(DATA_DIR / "rel_cleaned.csv")

    df_ner = pd.read_csv(ner_path)
    df_rel = pd.read_csv(rel_path)

    # FIX: normalize keys so "El Salvador" and "el salvador" both hit the same entry
    type_col = 'entity_type' if 'entity_type' in df_ner.columns else ('type' if 'type' in df_ner.columns else 'label')
    type_map = {
        str(e).lower().strip(): str(t).replace(" ", "_").upper()
        for e, t in zip(df_ner['entity'], df_ner[type_col])
    }

    # Build canonical id -> (name, type) map, first-seen wins
    id_to_meta = {}
    for _, row in df_rel.iterrows():
        for raw in [row['head'], row['tail']]:
            nid = normalize_id(raw)
            if nid not in id_to_meta:
                # lookup using normalized key — matches type_map normalization
                raw_type = type_map.get(str(raw).lower().strip(), 'ENTITY')
                id_to_meta[nid] = {'id': nid, 'name': str(raw), 'type': raw_type}

    sample = list(id_to_meta.items())[:5]
    for k, v in sample:
        print(k, v)

    node_data = list(id_to_meta.values())

    rel_data = [
        {
            'h_id': normalize_id(row['head']),
            't_id': normalize_id(row['tail']),
            'rel_type': str(row['relation']).replace(" ", "_").upper(),
            'chunk_id': str(row['article_chunk_id'])
        }
        for _, row in df_rel.iterrows()
    ]

    node_query = """
UNWIND $batch as row
CALL apoc.merge.node([row.type, 'Entity'], {id: row.id}, {name: row.name, type: row.type}, {}) YIELD node
RETURN count(*)
"""

    rel_query = """
    UNWIND $batch as row
    MATCH (h:Entity {id: row.h_id})
    MATCH (t:Entity {id: row.t_id})
    CALL apoc.merge.relationship(h, row.rel_type, {}, {chunk_id: row.chunk_id}, t) YIELD rel
    RETURN count(*)
    """

    batch_size = 5000
    driver = GraphDatabase.driver(URI, auth=(USER, PWD))

    with driver.session() as session:
        print("Wiping existing data...")
        session.run("MATCH (n) DETACH DELETE n")

        print("Setting constraints...")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE")

        print(f"Upserting {len(node_data)} unique nodes...")
        for i in range(0, len(node_data), batch_size):
            session.run(node_query, batch=node_data[i:i+batch_size])
            print(f"  Nodes: {min(i+batch_size, len(node_data))} / {len(node_data)}")

        print(f"Upserting {len(rel_data)} relationships...")
        for i in range(0, len(rel_data), batch_size):
            session.run(rel_query, batch=rel_data[i:i+batch_size])
            print(f"  Rels: {min(i+batch_size, len(rel_data))} / {len(rel_data)}")

    driver.close()
    print("Neo4j ingestion complete.")

if __name__ == "__main__":
    run_neo4j_rel_ingestion()

In [ ]:
# Clean up

import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PWD = os.getenv("NEO4J_PASSWORD")

def purge_noise():
    driver = GraphDatabase.driver(URI, auth=(USER, PWD))
    
    # 1. Patterns that are biologically/physically impossible
    # (e.g., A Person is not a Location for an Organization)
    type_mismatch_query = """
    MATCH (org:Entity)-[r:LOCATED_IN]->(p:Entity)
    WHERE p.type IN ['PERSON', 'PEOPLE'] OR org.type = 'PERSON'
    DELETE r
    RETURN count(*) as deleted_count
    """

    # 2. Known Hallucinations (Specific Cleanup)
    specific_noise_query = """
    MATCH (n:Entity {name: 'Nvidia'})-[r:COMPETES_WITH]->(c:Entity {name: "Denny'S"})
    DELETE r
    RETURN count(*) as deleted_count
    """

    # 3. Orphan Cleanup (Remove nodes with no connections left after purge)
    orphan_cleanup_query = """
    MATCH (n:Entity)
    WHERE NOT (n)--()
    DELETE n
    RETURN count(*) as deleted_count
    """

    with driver.session() as session:
        print("Starting Graph Sanitization...")
        
        # Run Type Mismatch Purge
        res1 = session.run(type_mismatch_query).single()
        print(f"  Removed {res1['deleted_count']} Type-Mismatched (Person-as-Location) relationships.")
        
        # Run Specific Noise Purge
        res2 = session.run(specific_noise_query).single()
        print(f"  Removed {res2['deleted_count']} known hallucinated relationships (Denny's).")
        
        # Run Orphan Cleanup
        res3 = session.run(orphan_cleanup_query).single()
        print(f"  Deleted {res3['deleted_count']} orphan nodes.")

    driver.close()
    print("Graph Sanitization Complete.")

if __name__ == "__main__":
    purge_noise()

In [ ]:
# Main snippet to extract entities and relationships from them using GLiNER

%%time
import torch
import pandas as pd
from gliner import GLiNER
from tqdm import tqdm

# ── Schema ────────────────────────────────────────────────────────────────────
ENTITY_LABELS = [
    "organization",
    "person",
    "product",
    "technology",
    "location",
    "market metric",
]

RELATION_LABELS = [
    "supplies",
    "competes with",
    "produces",
    "located in",
]

# ── Config ────────────────────────────────────────────────────────────────────
BATCH_SIZE    = 32
NER_THRESHOLD = 0.5
REL_THRESHOLD = 0.75

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = GLiNER.from_pretrained("knowledgator/gliner-multitask-large-v0.5").to(device)
model.eval()

# ── Prep ──────────────────────────────────────────────────────────────────────
records = df[["article_chunk_id", "chunk_text", "chunk_id"]].to_dict("records")
texts   = [r["chunk_text"] for r in records]

ner_results = []
rel_results = []
all_entities = []

# ── NER — batched with progress bar ──────────────────────────────────────────
print("Running NER...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="NER batches"):
        batch_texts = texts[i : i + BATCH_SIZE]
        batch_out = model.inference(
            batch_texts,
            labels=ENTITY_LABELS,
            threshold=NER_THRESHOLD,
            batch_size=BATCH_SIZE,
            flat_ner=True,
        )
        all_entities.extend(batch_out)

for row, entities in zip(records, all_entities):
    for ent in entities:
        ner_results.append({
            "article_chunk_id": row["article_chunk_id"],
            "chunk_index":      row["chunk_id"],
            "entity":           ent["text"],
            "type":             ent["label"].upper(),
            "start":            ent.get("start"),
            "end":              ent.get("end"),
            "confidence":       ent.get("score"),
        })

print(f"NER complete — {len(ner_results)} entities extracted")

# ── Relations — batched at chunk level ───────────────────────────────────────
print("Building relation prompts...")
chunk_prompts = []
for row, entities in zip(records, all_entities):
    if len(entities) < 2:
        continue
    prompts = [f"{e['text']} >> {rel}" for e in entities for rel in RELATION_LABELS]
    chunk_prompts.append((row, prompts))

print(f"Running relation extraction on {len(chunk_prompts)} chunks...")

with torch.no_grad():
    for i in tqdm(range(0, len(chunk_prompts), BATCH_SIZE), desc="Relation batches"):
        batch = chunk_prompts[i : i + BATCH_SIZE]
        for row, prompts in batch:
            try:
                out = model.inference(
                    row["chunk_text"],
                    labels=prompts,
                    threshold=REL_THRESHOLD,
                    flat_ner=True,
                )
                rels = out[0]  # unwrap List[List[Dict]]
                for rel in rels:
                    parts = rel["label"].split(" >> ", 1)
                    if len(parts) == 2:
                        rel_results.append({
                            "article_chunk_id": row["article_chunk_id"],
                            "chunk_index":      row["chunk_id"],
                            "head":             parts[0],
                            "relation":         parts[1].upper(),
                            "tail":             rel["text"],
                            "confidence":       rel.get("score"),
                        })
            except Exception as e:
                print(f"Relation failed for {row['article_chunk_id']}: {e}")

# ── Output ────────────────────────────────────────────────────────────────────
ner_df = pd.DataFrame(ner_results)
rel_df = pd.DataFrame(rel_results)

print(f"\n{len(ner_df)} entities | {len(rel_df)} relations across {len(df)} chunks")

ner_df.to_csv("/content/drive/MyDrive/ner_final_results.csv", index=False)  # fix: was ner_results.to_csv
rel_df.to_csv("/content/drive/MyDrive/rel_final_results.csv", index=False)  # fix: was rel_results.to_csv
print(f"NER checkpoint saved — {len(ner_df)} entities")
print(f"REL checkpoint saved — {len(rel_df)} rels")

display(ner_df.head())
display(rel_df.head())